In [ ]:
# Install the libraries required for our project
!pip install requests beautifulsoup4 pandas numpy matplotlib seaborn

In [ ]:
# Web scraping libraries
import requests
from bs4 import BeautifulSoup

# Data manipulation and numerical operations
import pandas as pd
import numpy as np

# Regular expressions and scraping delays
import re
import time
import random

print("All libraries imported successfully!")

Libraries imported successfully!


Define brands and cities

In [ ]:
# Car brands we want to analyse
brands = [
    "toyota",
    "hyundai",
    "tata",
    "honda"
]

# Cities we want to analyse
cities = [
    "hyderabad",
    "bangalore",
    "chennai",
    "delhi"
]

# Calculate the total number of brand-city combinations
total_combinations = len(brands) * len(cities)

print("Brands:", brands)
print("Cities:", cities)
print("Total URL combinations:", total_combinations)


Brands: ['toyota', 'hyundai', 'tata', 'honda']
Cities: ['hyderabad', 'bangalore', 'chennai', 'delhi']
Total URL combinations: 16


Generate the URLs dynamically

In [ ]:
# Generate one URL for every brand-city combination
urls = []

for brand in brands:
    for city in cities:

        # Construct the URL using the current brand and city
        url = f"https://www.cars24.com/buy-used-{brand}-cars-{city}/"

        # Save the generated URL
        urls.append({
            "Brand": brand,
            "City": city,
            "URL": url
        })

# Convert the list of dictionaries into a DataFrame
url_df = pd.DataFrame(urls)

# Display the generated URLs
display(url_df)

,Brand,City,URL
0,toyota,hyderabad,https://www.cars24.com/buy-used-toyota-cars-hy...
1,toyota,bangalore,https://www.cars24.com/buy-used-toyota-cars-ba...
2,toyota,chennai,https://www.cars24.com/buy-used-toyota-cars-ch...
3,toyota,delhi,https://www.cars24.com/buy-used-toyota-cars-de...
4,hyundai,hyderabad,https://www.cars24.com/buy-used-hyundai-cars-h...
5,hyundai,bangalore,https://www.cars24.com/buy-used-hyundai-cars-b...
6,hyundai,chennai,https://www.cars24.com/buy-used-hyundai-cars-c...
7,hyundai,delhi,https://www.cars24.com/buy-used-hyundai-cars-d...
8,tata,hyderabad,https://www.cars24.com/buy-used-tata-cars-hyde...
9,tata,bangalore,https://www.cars24.com/buy-used-tata-cars-bang...


Configure HTTP headers

In [ ]:
# Define HTTP headers for our web requests
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

print("HTTP headers configured successfully!")

HTTP headers configured successfully!


Send a request to one Cars24 page

In [ ]:
# Select the first URL from our generated URL table
test_url = url_df.loc[0, "URL"]

print("Testing URL:", test_url)

try:
    # Send an HTTP GET request with headers
    response = requests.get(
        test_url,
        headers=headers,
        timeout=20
    )

    # Display the HTTP status code
    print("Status code:", response.status_code)

    # Display the content type returned by the server
    print("Content type:", response.headers.get("Content-Type"))

    # Display the size of the HTML response
    print("Response size:", len(response.text), "characters")

except requests.RequestException as error:
    # Handle connection errors and timeouts
    print("Request failed:", error)

Testing URL: https://www.cars24.com/buy-used-toyota-cars-hyderabad/
Status code: 200
Content type: text/html; charset=utf-8
Response size: 1605218 characters


Parse and inspect the HTML

In [ ]:
# Check whether the previous request returned usable HTML
if (
    "response" in globals()
    and response.status_code == 200
    and "html" in response.headers.get("Content-Type", "").lower()
):
    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # Display the webpage title
    print("Page title:")
    print(soup.title.get_text(strip=True) if soup.title else "No title found")

    # Extract visible text from the HTML
    page_text = soup.get_text(" ", strip=True)

    # Display a sample of the page text
    print("\nFirst 3000 characters of page text:")
    print(page_text[:3000])

else:
    print(
        "Usable HTML was not received. "
        "Check the status code and content type before continuing."
    )

Page title:
61 Used Toyota Cars in Hyderabad, Second Hand Toyota Cars starting from ₹1.91 lakh – Cars24

First 3000 characters of page text:
61 Used Toyota Cars in Hyderabad, Second Hand Toyota Cars starting from ₹1.91 lakh – Cars24 Hyderabad Buy used car Buy Used Cars in Hyderabad Change city Browse by Model Browse by Make Browse by Price Browse by Body Type Browse by Fuel Type Browse by Transmission Browse by City Browse by Model Used Hyundai i10 Cars in Hyderabad Used Hyundai i20 Cars in Hyderabad Used Maruti Swift Cars in Hyderabad Used Tata NEXON Cars in Hyderabad Used Maruti Baleno Cars in Hyderabad Used KIA SELTOS Cars in Hyderabad Used Maruti Wagon R Cars in Hyderabad Used Renault Kwid Cars in Hyderabad Used Honda City Cars in Hyderabad Used Tata Tiago Cars in Hyderabad Buy cars online Browse by Make Buy cars online Browse by Price Used Cars Under 2 lakhs in Hyderabad Used Cars Under 3 lakhs in Hyderabad Used Cars Under 4 lakhs in Hyderabad Used Cars Under 5 lakhs in Hyderabad 

In [ ]:
# Check whether the page contains common listing-related keywords
keywords = [
    "Toyota",
    "Petrol",
    "Diesel",
    "Automatic",
    "Manual",
    "km",
    "₹"
]

print("Keyword occurrence check:\n")

for keyword in keywords:
    # Count how many times each keyword occurs in the HTML
    count = response.text.lower().count(keyword.lower())

    print(f"{keyword}: {count}")

Keyword occurrence check:

Toyota: 840
Petrol: 109
Diesel: 72
Automatic: 40
Manual: 90
km: 85
₹: 162


In [ ]:
# Parse the HTML response into a BeautifulSoup object
soup = BeautifulSoup(response.text, "html.parser")

# Extract visible text and normalize whitespace
page_text = soup.get_text(" ", strip=True)
page_text = re.sub(r"\s+", " ", page_text).strip()

# Print a sample of the extracted text
print("Total visible text characters:", len(page_text))
print("\nSample webpage text:\n")
print(page_text[:5000])

Total visible text characters: 18600

Sample webpage text:

61 Used Toyota Cars in Hyderabad, Second Hand Toyota Cars starting from ₹1.91 lakh – Cars24 Hyderabad Buy used car Buy Used Cars in Hyderabad Change city Browse by Model Browse by Make Browse by Price Browse by Body Type Browse by Fuel Type Browse by Transmission Browse by City Browse by Model Used Hyundai i10 Cars in Hyderabad Used Hyundai i20 Cars in Hyderabad Used Maruti Swift Cars in Hyderabad Used Tata NEXON Cars in Hyderabad Used Maruti Baleno Cars in Hyderabad Used KIA SELTOS Cars in Hyderabad Used Maruti Wagon R Cars in Hyderabad Used Renault Kwid Cars in Hyderabad Used Honda City Cars in Hyderabad Used Tata Tiago Cars in Hyderabad Buy cars online Browse by Make Buy cars online Browse by Price Used Cars Under 2 lakhs in Hyderabad Used Cars Under 3 lakhs in Hyderabad Used Cars Under 4 lakhs in Hyderabad Used Cars Under 5 lakhs in Hyderabad Used Cars Under 7 lakhs in Hyderabad Used Cars Under 8 lakhs in Hyderabad Used Ca

In [ ]:
import json

# Find script elements that contain JSON-LD structured data
jsonld_scripts = soup.find_all(
    "script",
    type="application/ld+json"
)

print("Number of JSON-LD script elements:", len(jsonld_scripts))

# Inspect the first few structured-data elements
for index, script in enumerate(jsonld_scripts[:3], start=1):
    print(f"\n--- JSON-LD element {index} ---")

    try:
        # Convert the JSON text into a Python object
        structured_data = json.loads(script.string or script.get_text())

        # Print a short preview
        print(str(structured_data)[:2000])

    except (json.JSONDecodeError, TypeError) as error:
        print("Could not parse this element:", error)

Number of JSON-LD script elements: 4

--- JSON-LD element 1 ---
[{'@context': 'https://schema.org/', '@id': 'https://www.cars24.com/', 'url': 'https://www.cars24.com/', 'name': 'Cars24', '@type': 'WebPage'}, {'@context': 'https://schema.org/', '@id': 'https://www.cars24.com/', 'url': 'https://www.cars24.com/', 'name': 'Cars24', '@type': 'Website'}]

--- JSON-LD element 2 ---
{'@context': 'http://schema.org', '@type': 'BreadcrumbList', 'itemListElement': [{'@type': 'ListItem', 'position': 1, 'item': {'@id': 'https://www.cars24.com/', 'name': 'Home'}}, {'@type': 'ListItem', 'position': 2, 'item': {'@id': 'https://www.cars24.com/buy-used-cars/', 'name': 'Used Cars'}}, {'@type': 'ListItem', 'position': 3, 'item': {'@id': 'https://www.cars24.com/buy-used-cars-hyderabad/', 'name': 'Used Cars in Hyderabad'}}, {'@type': 'ListItem', 'position': 4, 'item': {'name': 'Used Toyota Cars in Hyderabad'}}]}

--- JSON-LD element 3 ---
{'@context': 'https://schema.org/', '@graph': [{'@type': 'ListItem', 

In [ ]:
# Search selected HTML elements for possible listing information
candidate_tags = ["article", "li", "div"]

# Keywords that may appear in a car listing
listing_keywords = ["petrol", "diesel", "km", "automatic", "manual"]

matches = []

for tag_name in candidate_tags:
    # Find elements of the current tag type
    for element in soup.find_all(tag_name):
        text_content = element.get_text(" ", strip=True)

        # Ignore very short or extremely large text blocks
        if not (30 <= len(text_content) <= 1200):
            continue

        # Keep elements containing at least two listing-related keywords
        text_lower = text_content.lower()
        keyword_matches = sum(
            keyword in text_lower
            for keyword in listing_keywords
        )

        if keyword_matches >= 2:
            matches.append({
                "Tag": tag_name,
                "Class": element.get("class"),
                "Text": text_content
            })

# Display a sample of candidate elements
print("Candidate elements found:", len(matches))

for item in matches[:10]:
    print("\nTag:", item["Tag"])
    print("Class:", item["Class"])
    print("Text:", item["Text"][:500])

Candidate elements found: 94

Tag: div
Class: ['sc-kSGOQT', 'kgiiSF']
Text: Browse by Fuel Type Petrol Diesel CNG Electric Hybrid Buy cars online

Tag: div
Class: ['sc-jsTgWx', 'eTzcaI']
Text: Petrol Diesel CNG Electric Hybrid Buy cars online

Tag: div
Class: ['sc-cLNono', 'gLZnUN']
Text: Petrol Diesel CNG Electric Hybrid

Tag: div
Class: ['sc-kSGOQT', 'kgiiSF']
Text: Browse by Transmission Manual Automatic Buy cars online

Tag: div
Class: ['sc-jsTgWx', 'eTzcaI']
Text: Manual Automatic Buy cars online

Tag: div
Class: ['styles_normalCardWrapper__qDZjq']
Text: Cars24 Owned Stock 2021 Toyota Glanza G CVT 70,609 km Petrol Auto TS-07 EMI ₹9,792/m* ₹5.55 lakh Includes RC transfer & more Bachupally, Hyderabad

Tag: div
Class: ['styles_contentWrap__9oSrl']
Text: Cars24 Owned Stock 2021 Toyota Glanza G CVT 70,609 km Petrol Auto TS-07 EMI ₹9,792/m* ₹5.55 lakh Includes RC transfer & more Bachupally, Hyderabad

Tag: div
Class: ['styles_outer__NTVth']
Text: 2021 Toyota Glanza G CVT 70,609 km Petro

In [ ]:

# Find scripts that may contain embedded listing data
scripts = soup.find_all("script")

print("Total script elements:", len(scripts))

# Search script contents for relevant keywords
search_terms = [
    "kilometers",
    "transmission",
    "fuelType",
    "price",
    "modelYear"
]

for term in search_terms:
    count = sum(
        term.lower() in (script.string or script.get_text()).lower()
        for script in scripts
    )

    print(f"Scripts containing '{term}': {count}")


Total script elements: 94
Scripts containing 'kilometers': 0
Scripts containing 'transmission': 2
Scripts containing 'fuelType': 2
Scripts containing 'price': 8
Scripts containing 'modelYear': 0


In [ ]:

# Inspect scripts containing multiple car-related terms
for index, script in enumerate(scripts):
    script_text = script.string or script.get_text()

    # Count how many target terms appear in the script
    matches = sum(
        term.lower() in script_text.lower()
        for term in search_terms
    )

    # Display scripts containing at least two relevant terms
    if matches >= 2:
        print(f"\nScript index: {index}")
        print("Script length:", len(script_text))
        print("Preview:")
        print(script_text[:1500])



Script index: 41
Script length: 332635
Preview:
self.__next_f.push([1,"23:[\"$\",\"$L2a\",null,{\"layoutData\":{\"carList\":{\"filtersAndSortOrder\":{\"sortOrder\":\"P\"},\"content\":[],\"similarCars\":{},\"isSSRSimilarCar\":false,\"metaContent\":{\"metaTitle\":null,\"metaDescription\":null,\"metaKeyWord\":null,\"heading\":null},\"interLinkingUrls\":null,\"filters\":null,\"unappliedOptions\":{},\"appliedOptions\":{\"sort\":\"bestmatch\",\"serveWarrantyCount\":true},\"cities\":{},\"filterCollapsibleStates\":null,\"isFilterSettingsFetched\":false,\"totalCars\":0,\"totalPages\":null,\"activeScreen\":\"listing\",\"activeTab\":null,\"error\":null,\"displayType\":null,\"isSSR\":false,\"page\":0,\"pageSize\":20,\"isCarCountLoading\":false,\"lastUpdatedAppliedFilters\":null,\"lastSelectedFilterKey\":null,\"unappliedCarCount\":null,\"lastUpdatedUnappliedFilters\":null,\"isRedirected\":false,\"isLoading\":false,\"listName\":\"Buy_used_cars_C2C_Car_Listing_Page\",\"showWarranty\":false,\"offScre

In [ ]:

# Find text-bearing HTML elements that might represent car listings
candidate_elements = []

for element in soup.find_all(["article", "li", "div"]):
    text_content = element.get_text(" ", strip=True)
    text_lower = text_content.lower()

    # Look for a rupee symbol and vehicle-related information
    has_price = "₹" in text_content
    has_vehicle_detail = any(
        term in text_lower
        for term in ["petrol", "diesel", "km", "automatic", "manual"]
    )

    # Keep reasonably sized candidate elements
    if has_price and has_vehicle_detail and 50 <= len(text_content) <= 1000:
        candidate_elements.append(element)

print("Candidate elements:", len(candidate_elements))

# Print a small sample for manual inspection
for index, element in enumerate(candidate_elements[:5], start=1):
    print(f"\nCandidate {index}")
    print("Tag:", element.name)
    print("Classes:", element.get("class"))
    print("Text:", element.get_text(" ", strip=True)[:400])


Candidate elements: 63

Candidate 1
Tag: div
Classes: None
Text: All Luxe Budget ₹ 50,000 ₹ 75,00,000 Minimum Maximum Make & Model All Brands Swift NEXON Baleno SELTOS Creta HECTOR Maruti ( 530 ) Hyundai ( 393 ) Tata ( 239 ) Mahindra ( 138 ) Honda ( 119 ) KIA ( 118 ) Renault ( 82 ) MG ( 82 ) Volkswagen ( 76 ) Ford ( 75 ) Toyota ( 61 ) Skoda ( 59 ) Nissan ( 45 ) Datsun ( 19 ) Mercedes Benz ( 16 ) Audi ( 15 ) BMW ( 13 ) Jeep ( 11 ) Landrover ( 7 ) CITROEN ( 6 ) V

Candidate 2
Tag: div
Classes: ['styles_filterContainer__DZYue', 'styles_filterLuxeContainer__5DhV9']
Text: Budget ₹ 50,000 ₹ 75,00,000 Minimum Maximum Make & Model All Brands Swift NEXON Baleno SELTOS Creta HECTOR Maruti ( 530 ) Hyundai ( 393 ) Tata ( 239 ) Mahindra ( 138 ) Honda ( 119 ) KIA ( 118 ) Renault ( 82 ) MG ( 82 ) Volkswagen ( 76 ) Ford ( 75 ) Toyota ( 61 ) Skoda ( 59 ) Nissan ( 45 ) Datsun ( 19 ) Mercedes Benz ( 16 ) Audi ( 15 ) BMW ( 13 ) Jeep ( 11 ) Landrover ( 7 ) CITROEN ( 6 ) Volvo ( 6 

Candidate 3
Tag: div
Cla

In [ ]:

# Select a candidate that looks like an actual car listing
listing = candidate_elements[3]  # Candidate 4 from our previous output

# Display the HTML structure of this listing
print(listing.prettify()[:10000])


<div class="styles_normalCardWrapper__qDZjq">
 <div class="sc-dkBdzd bUodwL">
  <div class="sc-evzXkY kiqjhR">
   <div class="sc-fHSyaj eATnRu">
    <img alt="2021 Toyota Glanza - Hatchback - Petrol - Automatic - ₹5.55 lakh" class="shrinkOnTouch" data-nimg="1" decoding="async" height="120" loading="lazy" src="https://media.cars24.com/hello-ar/dev/transformed/uploads/no_bg/1ca63a56-70b2-11f1-bd55-02ede2007fbe/m/66ca410d-5620-4643-9760-d60e5fe2ee30/096dab4b-fa6d-469b-bdc2-a34e7c4ca82b/74baee8b-6d75-4baa-8ae3-73a1580d3f7e/slot/1.png?w=200&amp;format=auto&amp;dpr=2&amp;pad=48,0,0,0&amp;trim-color=auto" style="color:transparent" width="220"/>
   </div>
   <div class="sc-dubCtS fWYOjo">
    <div class="styles_outer__ZH1Cg">
     <button>
      <img alt="Add to wishlist" class="styles_icon__vn_BC shrinkOnTouch" data-nimg="1" decoding="async" height="24" loading="lazy" src="https://assets.cars24.com/production/india-website/catalog/261009052914/_next/static/media/icon-heart-empty.479d2b8c.svg"

In [ ]:

# Find elements whose class name contains "normalCardWrapper"
listing_cards = soup.select(
    'div[class*="normalCardWrapper"]'
)

print("Listing containers found:", len(listing_cards))

# Display the text from the first five containers
for index, card in enumerate(listing_cards[:5], start=1):
    print(f"\n--- Listing {index} ---")
    print(card.get_text(" ", strip=True))


Listing containers found: 20

--- Listing 1 ---
Cars24 Owned Stock 2021 Toyota Glanza G CVT 70,609 km Petrol Auto TS-07 EMI ₹9,792/m* ₹5.55 lakh Includes RC transfer & more Bachupally, Hyderabad

--- Listing 2 ---
Cars24 Owned Stock 2020 Toyota Glanza V CVT 86,387 km Petrol Auto TS-15 EMI ₹9,435/m* ₹5.68L ₹5.34 lakh Includes RC transfer & more GVK One Mall, Banjara Hills, Hyderabad, Telangana

--- Listing 3 ---
Cars24 Owned Stock 2021 Toyota Glanza V 1,05,485 km Petrol Manual AP-39 EMI ₹12,252/m* ₹5.94L ₹5.51 lakh Includes RC transfer & more Upperpally, Attapur, Hyderabad

--- Listing 4 ---
Cars24 Owned Stock 2025 Toyota Glanza V AMT 19,742 km Petrol Auto TG-08 EMI ₹15,818/m* ₹9.68L ₹9.24 lakh Includes RC transfer & more Kompally, Hyderabad

--- Listing 5 ---
Cars24 Owned Stock 2018 Toyota YARIS G CVT 74,307 km Petrol Auto TS-11 EMI ₹9,380/m* ₹5.65L ₹5.31 lakh Includes RC transfer & more Kompally, Hyderabad


In [ ]:

def extract_car_details(card, brand, city, source_url):
    """
    Extract available car details from one Cars24 listing.
    Missing or ambiguous fields are kept as None.
    """

    # Extract all text from the listing container
    text = card.get_text(" ", strip=True)

    # Normalize repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Extract model year from the listing text
    year_match = re.search(r"\b(19\d{2}|20\d{2})\b", text)
    model_year = int(year_match.group(1)) if year_match else None

    # Extract kilometres, allowing commas in the number
    km_match = re.search(
        r"\b([\d,]+)\s*(?:km|kms|kilometres|kilometers)\b",
        text,
        flags=re.IGNORECASE
    )
    kilometers = (
        int(re.sub(r"[^\d]", "", km_match.group(1)))
        if km_match else None
    )

    # Extract price in lakh or crore, if present
    price_match = re.search(
        r"₹\s*([\d,.]+)\s*(lakh|lac|crore|cr)\b",
        text,
        flags=re.IGNORECASE
    )

    price_rupees = None

    if price_match:
        amount = float(price_match.group(1).replace(",", ""))
        unit = price_match.group(2).lower()

        # Convert displayed Indian price units into rupees
        multiplier = 10_000_000 if unit in ["crore", "cr"] else 100_000
        price_rupees = amount * multiplier

    # Identify fuel type using the available listing text
    fuel_match = re.search(
        r"\b(Petrol|Diesel|CNG|Electric|Hybrid)\b",
        text,
        flags=re.IGNORECASE
    )
    fuel_type = fuel_match.group(1).title() if fuel_match else None

    # Identify transmission
    transmission_match = re.search(
        r"\b(Automatic|Auto|Manual)\b",
        text,
        flags=re.IGNORECASE
    )

    transmission = None

    if transmission_match:
        value = transmission_match.group(1).lower()
        transmission = "Automatic" if value in ["automatic", "auto"] else "Manual"

    # Find the first element that appears to contain the car's name
    car_name = None

    for element in card.find_all(["h2", "h3", "h4", "a"]):
        candidate_name = element.get_text(" ", strip=True)

        if year_match and year_match.group(1) in candidate_name:
            car_name = candidate_name
            break

    # Return extracted values and source information
    return {
        "Car_Name": car_name,
        "Brand": brand.title(),
        "City": city.title(),
        "Model_Year": model_year,
        "Kilometers_Driven": kilometers,
        "Fuel_Type": fuel_type,
        "Transmission": transmission,
        "Price": price_rupees,
        "Owner_Type": None,  # Populate only after verifying owner information
        "URL": source_url
    }


# Test the extraction function on one listing
test_record = extract_car_details(
    listing_cards[0],
    brand="toyota",
    city="hyderabad",
    source_url=test_url
)

print(test_record)


{'Car_Name': None, 'Brand': 'Toyota', 'City': 'Hyderabad', 'Model_Year': 2021, 'Kilometers_Driven': 70609, 'Fuel_Type': 'Petrol', 'Transmission': 'Automatic', 'Price': 555000.0, 'Owner_Type': None, 'URL': 'https://www.cars24.com/buy-used-toyota-cars-hyderabad/'}


In [ ]:

# Select the first car listing
card = listing_cards[0]

# Inspect heading and link elements for the car name
for element in card.find_all(["h1", "h2", "h3", "h4", "a", "p", "span"]):
    text = element.get_text(" ", strip=True)

    # Print only elements containing a four-digit year
    if re.search(r"\b(19\d{2}|20\d{2})\b", text):
        print("Tag:", element.name)
        print("Class:", element.get("class"))
        print("Text:", text[:200])
        print("-" * 50)


Tag: span
Class: ['sc-ksBlki', 'dxpZAa']
Text: 2021 Toyota Glanza
--------------------------------------------------


In [ ]:

# Print the complete HTML for the first listing
print(card.prettify()[:8000])

# Inspect links within the listing
print("\nLinks found in the listing:")

for link in card.find_all("a", href=True):
    print({
        "text": link.get_text(" ", strip=True),
        "href": link["href"]
    })


<div class="styles_normalCardWrapper__qDZjq">
 <div class="sc-dkBdzd bUodwL">
  <div class="sc-evzXkY kiqjhR">
   <div class="sc-fHSyaj eATnRu">
    <img alt="2021 Toyota Glanza - Hatchback - Petrol - Automatic - ₹5.55 lakh" class="shrinkOnTouch" data-nimg="1" decoding="async" height="120" loading="lazy" src="https://media.cars24.com/hello-ar/dev/transformed/uploads/no_bg/1ca63a56-70b2-11f1-bd55-02ede2007fbe/m/66ca410d-5620-4643-9760-d60e5fe2ee30/096dab4b-fa6d-469b-bdc2-a34e7c4ca82b/74baee8b-6d75-4baa-8ae3-73a1580d3f7e/slot/1.png?w=200&amp;format=auto&amp;dpr=2&amp;pad=48,0,0,0&amp;trim-color=auto" style="color:transparent" width="220"/>
   </div>
   <div class="sc-dubCtS fWYOjo">
    <div class="styles_outer__ZH1Cg">
     <button>
      <img alt="Add to wishlist" class="styles_icon__vn_BC shrinkOnTouch" data-nimg="1" decoding="async" height="24" loading="lazy" src="https://assets.cars24.com/production/india-website/catalog/261009052914/_next/static/media/icon-heart-empty.479d2b8c.svg"

In [ ]:

def extract_car_details(card, brand, city, source_url):
    """Extract car details from one Cars24 listing card."""

    # Extract and normalize the listing text
    text = re.sub(r"\s+", " ", card.get_text(" ", strip=True)).strip()

    # 1. Extract the model year
    year_match = re.search(r"\b(19\d{2}|20\d{2})\b", text)
    model_year = int(year_match.group(1)) if year_match else None

    # 2. Extract the title and variant from the listing's title container
    title_element = card.select_one('div[class*="sc-bWOGAF"]')

    car_name = None

    if title_element:
        title_parts = [
            element.get_text(" ", strip=True)
            for element in title_element.find_all("span", recursive=False)
        ]

        # Combine the model title and variant, if both exist
        if title_parts:
            car_name = " ".join(title_parts)

    # Fallback: use the image alt text when a title is unavailable
    if not car_name:
        image = card.find("img", alt=re.compile(r"\b(19\d{2}|20\d{2})\b"))
        if image:
            car_name = image.get("alt", "").split(" - ")[0].strip() or None

    # 3. Extract kilometres driven
    km_match = re.search(
        r"\b([\d,]+)\s*(?:km|kms|kilometres|kilometers)\b",
        text,
        re.IGNORECASE
    )
    kilometers = (
        int(re.sub(r"\D", "", km_match.group(1)))
        if km_match else None
    )

    # 4. Extract price and convert lakh/crore to rupees
    price_match = re.search(
        r"₹\s*([\d,.]+)\s*(lakh|lac|crore|cr)\b",
        text,
        re.IGNORECASE
    )

    price_rupees = None

    if price_match:
        amount = float(price_match.group(1).replace(",", ""))
        unit = price_match.group(2).lower()

        multiplier = (
            10_000_000 if unit in ["crore", "cr"]
            else 100_000
        )
        price_rupees = amount * multiplier

    # 5. Extract fuel type
    fuel_match = re.search(
        r"\b(Petrol|Diesel|CNG|Electric|Hybrid)\b",
        text,
        re.IGNORECASE
    )
    fuel_type = fuel_match.group(1).title() if fuel_match else None

    # 6. Extract transmission
    transmission_match = re.search(
        r"\b(Automatic|Auto|Manual)\b",
        text,
        re.IGNORECASE
    )

    transmission = None
    if transmission_match:
        transmission = (
            "Automatic"
            if transmission_match.group(1).lower() in ["auto", "automatic"]
            else "Manual"
        )

    # 7. Extract the listing location
    location_element = card.select_one('div[class*="hubAddress"]')
    location = (
        location_element.get_text(" ", strip=True)
        if location_element else None
    )

    # Return the extracted record
    return {
        "Car_Name": car_name,
        "Brand": brand.title(),
        "City": city.title(),
        "Model_Year": model_year,
        "Kilometers_Driven": kilometers,
        "Fuel_Type": fuel_type,
        "Transmission": transmission,
        "Price": price_rupees,
        "Owner_Type": None,
        "Location": location,
        "URL": source_url
    }


# Test the updated function on the first listing
test_record = extract_car_details(
    listing_cards[0],
    brand="toyota",
    city="hyderabad",
    source_url=test_url
)

print(test_record)


{'Car_Name': '2021 Toyota Glanza G CVT', 'Brand': 'Toyota', 'City': 'Hyderabad', 'Model_Year': 2021, 'Kilometers_Driven': 70609, 'Fuel_Type': 'Petrol', 'Transmission': 'Automatic', 'Price': 555000.0, 'Owner_Type': None, 'Location': 'Bachupally, Hyderabad', 'URL': 'https://www.cars24.com/buy-used-toyota-cars-hyderabad/'}


In [ ]:

# Extract details from the first five available listing cards
sample_records = []

for card in listing_cards[:5]:
    record = extract_car_details(
        card=card,
        brand="toyota",
        city="hyderabad",
        source_url=test_url
    )
    sample_records.append(record)

# Convert the records into a DataFrame
sample_df = pd.DataFrame(sample_records)

# Display the extracted data
display(sample_df)


,Car_Name,Brand,City,Model_Year,Kilometers_Driven,Fuel_Type,Transmission,Price,Owner_Type,Location,URL
0,2021 Toyota Glanza G CVT,Toyota,Hyderabad,2021,70609,Petrol,Automatic,555000.0,None,"Bachupally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
1,2020 Toyota Glanza V CVT,Toyota,Hyderabad,2020,86387,Petrol,Automatic,534000.0,None,"GVK One Mall, Banjara Hills, Hyderabad, Telangana",https://www.cars24.com/buy-used-toyota-cars-hy...
2,2021 Toyota Glanza V,Toyota,Hyderabad,2021,105485,Petrol,Manual,551000.0,None,"Upperpally, Attapur, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
3,2025 Toyota Glanza V AMT,Toyota,Hyderabad,2025,19742,Petrol,Automatic,924000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
4,2018 Toyota YARIS G CVT,Toyota,Hyderabad,2018,74307,Petrol,Automatic,531000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...


In [ ]:

def scrape_cars_page(brand, city, url):
    """Fetch one Cars24 page and extract its available car listings."""

    print(f"\nScraping {brand.title()} - {city.title()}")

    try:
        # Send the request using our configured HTTP headers
        page_response = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        print("Status code:", page_response.status_code)

        # Do not process unsuccessful responses
        if page_response.status_code != 200:
            print("Page could not be scraped.")
            return []

        # Verify that the response contains HTML
        content_type = page_response.headers.get("Content-Type", "")

        if "html" not in content_type.lower():
            print("Response is not HTML.")
            return []

        # Parse the webpage HTML
        page_soup = BeautifulSoup(
            page_response.text,
            "html.parser"
        )

        # Locate car listing containers
        cards = page_soup.select(
            'div[class*="normalCardWrapper"]'
        )

        print("Listing containers found:", len(cards))

        # Extract details from each listing
        records = []

        for card in cards:
            record = extract_car_details(
                card=card,
                brand=brand,
                city=city,
                source_url=url
            )

            # Keep records with an identifiable car name
            if record["Car_Name"]:
                records.append(record)

        print("Valid named listings extracted:", len(records))

        return records

    except requests.RequestException as error:
        print("Request error:", error)
        return []


In [ ]:

# Store all extracted records here
all_records = []

# Store a summary of each URL's scraping results
scraping_summary = []

# Iterate through every generated brand-city URL
for _, row in url_df.iterrows():

    brand = row["Brand"]
    city = row["City"]
    url = row["URL"]

    # Scrape this brand-city page
    records = scrape_cars_page(
        brand=brand,
        city=city,
        url=url
    )

    # Add the extracted records to the overall collection
    all_records.extend(records)

    # Record how many listings were collected
    scraping_summary.append({
        "Brand": brand,
        "City": city,
        "URL": url,
        "Listings_Extracted": len(records)
    })

    # Wait between requests to reduce server load
    time.sleep(random.uniform(2, 4))

# Create a DataFrame from the collected records
df_raw = pd.DataFrame(all_records)

# Create a summary table
summary_df = pd.DataFrame(scraping_summary)

print("\nTotal listings extracted:", len(df_raw))

display(summary_df)



Scraping Toyota - Hyderabad
Status code: 200
Listing containers found: 20
Valid named listings extracted: 20

Scraping Toyota - Bangalore
Status code: 200
Listing containers found: 20
Valid named listings extracted: 20

Scraping Toyota - Chennai
Status code: 200
Listing containers found: 12
Valid named listings extracted: 12

Scraping Toyota - Delhi
Status code: 404
Page could not be scraped.

Scraping Hyundai - Hyderabad
Status code: 200
Listing containers found: 20
Valid named listings extracted: 20

Scraping Hyundai - Bangalore
Status code: 200
Listing containers found: 20
Valid named listings extracted: 20

Scraping Hyundai - Chennai
Status code: 200
Listing containers found: 20
Valid named listings extracted: 20

Scraping Hyundai - Delhi
Status code: 404
Page could not be scraped.

Scraping Tata - Hyderabad
Status code: 200
Listing containers found: 20
Valid named listings extracted: 20

Scraping Tata - Bangalore
Status code: 200
Listing containers found: 20
Valid named listings 

,Brand,City,URL,Listings_Extracted
0,toyota,hyderabad,https://www.cars24.com/buy-used-toyota-cars-hy...,20
1,toyota,bangalore,https://www.cars24.com/buy-used-toyota-cars-ba...,20
2,toyota,chennai,https://www.cars24.com/buy-used-toyota-cars-ch...,12
3,toyota,delhi,https://www.cars24.com/buy-used-toyota-cars-de...,0
4,hyundai,hyderabad,https://www.cars24.com/buy-used-hyundai-cars-h...,20
5,hyundai,bangalore,https://www.cars24.com/buy-used-hyundai-cars-b...,20
6,hyundai,chennai,https://www.cars24.com/buy-used-hyundai-cars-c...,20
7,hyundai,delhi,https://www.cars24.com/buy-used-hyundai-cars-d...,0
8,tata,hyderabad,https://www.cars24.com/buy-used-tata-cars-hyde...,20
9,tata,bangalore,https://www.cars24.com/buy-used-tata-cars-bang...,20


In [ ]:

# Display the number of rows and columns
print("Dataset shape:", df_raw.shape)

# Display column names
print("\nColumns:")
print(df_raw.columns.tolist())

# Preview the first ten records
display(df_raw.head(10))

# Count listings by brand and city
if not df_raw.empty:
    print("\nListings by brand:")
    display(df_raw["Brand"].value_counts())

    print("\nListings by city:")
    display(df_raw["City"].value_counts())
else:
    print("No listings were extracted. Check the scraping summary.")


Dataset shape: (232, 11)

Columns:
['Car_Name', 'Brand', 'City', 'Model_Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission', 'Price', 'Owner_Type', 'Location', 'URL']


,Car_Name,Brand,City,Model_Year,Kilometers_Driven,Fuel_Type,Transmission,Price,Owner_Type,Location,URL
0,2020 Toyota Glanza V CVT,Toyota,Hyderabad,2020,86387,Petrol,Automatic,534000.0,None,"GVK One Mall, Banjara Hills, Hyderabad, Telangana",https://www.cars24.com/buy-used-toyota-cars-hy...
1,2021 Toyota Glanza V,Toyota,Hyderabad,2021,105485,Petrol,Manual,551000.0,None,"Upperpally, Attapur, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
2,2021 Toyota Glanza G CVT,Toyota,Hyderabad,2021,70609,Petrol,Automatic,555000.0,None,"Bachupally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
3,2025 Toyota Glanza V AMT,Toyota,Hyderabad,2025,19742,Petrol,Automatic,924000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
4,2018 Toyota YARIS G CVT,Toyota,Hyderabad,2018,74307,Petrol,Automatic,531000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
5,2018 Toyota YARIS G CVT,Toyota,Hyderabad,2018,92328,Petrol,Automatic,482000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
6,2017 Toyota Corolla Altis GL DIESEL,Toyota,Hyderabad,2017,141572,Diesel,Manual,655000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
7,2022 Toyota URBAN CRUISER HYRYDER V HYBRID,Toyota,Hyderabad,2022,108490,Hybrid,Automatic,1247000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
8,2025 Toyota Urban Cruiser Taisor S 1.2 Petrol AMT,Toyota,Hyderabad,2025,8207,Petrol,Automatic,1471000.0,None,"Bachupally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
9,2013 Toyota Etios GD,Toyota,Hyderabad,2013,95534,Diesel,Manual,332000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...



Listings by brand:


,count
Brand,
Hyundai,60
Tata,60
Honda,60
Toyota,52



Listings by city:


,count
City,
Hyderabad,80
Bangalore,80
Chennai,72


In [ ]:

# Save the extracted dataset to a CSV file
if not df_raw.empty:
    df_raw.to_csv(
        "scraped_used_cars.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved: scraped_used_cars.csv")
else:
    print("Dataset is empty. CSV was not saved.")


Saved: scraped_used_cars.csv


In [ ]:

from google.colab import files

if not df_raw.empty:
    files.download("scraped_used_cars.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Create a separate copy of the raw dataset
df_raw_backup = df_raw.copy()

# Save the original scraped records to CSV
df_raw_backup.to_csv(
    "scraped_used_cars.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Raw dataset saved successfully!")
print("Rows:", len(df_raw_backup))
print("Columns:", len(df_raw_backup.columns))


Raw dataset saved successfully!
Rows: 232
Columns: 11


# Exploratory data analysis

In [ ]:

# Preserve the original scraped dataset
df_raw_backup = df_raw.copy(deep=True)

# Display dataset dimensions
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

# Check column data types
print("\nData types:")
print(df_raw.dtypes)

# Check missing values in each column
print("\nMissing values:")
display(df_raw.isnull().sum().to_frame("Missing_Count"))

# Check the number of duplicate rows
print("\nExact duplicate rows:", df_raw.duplicated().sum())


Rows: 232
Columns: 11

Data types:
Car_Name              object
Brand                 object
City                  object
Model_Year             int64
Kilometers_Driven      int64
Fuel_Type             object
Transmission          object
Price                float64
Owner_Type            object
Location              object
URL                   object
dtype: object

Missing values:


,Missing_Count
Car_Name,0
Brand,0
City,0
Model_Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Price,0
Owner_Type,232
Location,0



Exact duplicate rows: 0


In [ ]:

# Work on a separate DataFrame for cleaning
df_clean = df_raw.copy(deep=True)

# Text columns that need whitespace normalization
text_columns = [
    "Car_Name",
    "Brand",
    "City",
    "Fuel_Type",
    "Transmission",
    "Owner_Type",
    "Location"
]

for column in text_columns:
    # Convert values to strings while preserving missing values
    df_clean[column] = df_clean[column].map(
        lambda value: (
            re.sub(r"\s+", " ", value).strip()
            if isinstance(value, str)
            else value
        )
    )

# Standardize brand, city, fuel and transmission labels
for column in ["Brand", "City", "Fuel_Type", "Transmission"]:
    df_clean[column] = df_clean[column].str.title()

# Preview the cleaned text
display(df_clean.head())


,Car_Name,Brand,City,Model_Year,Kilometers_Driven,Fuel_Type,Transmission,Price,Owner_Type,Location,URL
0,2020 Toyota Glanza V CVT,Toyota,Hyderabad,2020,86387,Petrol,Automatic,534000.0,None,"GVK One Mall, Banjara Hills, Hyderabad, Telangana",https://www.cars24.com/buy-used-toyota-cars-hy...
1,2021 Toyota Glanza V,Toyota,Hyderabad,2021,105485,Petrol,Manual,551000.0,None,"Upperpally, Attapur, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
2,2021 Toyota Glanza G CVT,Toyota,Hyderabad,2021,70609,Petrol,Automatic,555000.0,None,"Bachupally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
3,2025 Toyota Glanza V AMT,Toyota,Hyderabad,2025,19742,Petrol,Automatic,924000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...
4,2018 Toyota YARIS G CVT,Toyota,Hyderabad,2018,74307,Petrol,Automatic,531000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...


In [ ]:

# Normalize whitespace in the car name
df_clean["Car_Name"] = (
    df_clean["Car_Name"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Extract a four-digit year from the car name
year_from_name = df_clean["Car_Name"].str.extract(
    r"\b(19\d{2}|20\d{2})\b",
    expand=False
)

# Use the extracted year when the existing year is missing
df_clean["Model_Year"] = (
    pd.to_numeric(df_clean["Model_Year"], errors="coerce")
    .fillna(pd.to_numeric(year_from_name, errors="coerce"))
)

# Display examples
display(df_clean[["Car_Name", "Model_Year"]].head(10))


,Car_Name,Model_Year
0,2020 Toyota Glanza V CVT,2020
1,2021 Toyota Glanza V,2021
2,2021 Toyota Glanza G CVT,2021
3,2025 Toyota Glanza V AMT,2025
4,2018 Toyota YARIS G CVT,2018
5,2018 Toyota YARIS G CVT,2018
6,2017 Toyota Corolla Altis GL DIESEL,2017
7,2022 Toyota URBAN CRUISER HYRYDER V HYBRID,2022
8,2025 Toyota Urban Cruiser Taisor S 1.2 Petrol AMT,2025
9,2013 Toyota Etios GD,2013


In [ ]:

# Convert kilometre values to strings temporarily
km_text = df_clean["Kilometers_Driven"].astype("string")

# Remove all characters except digits
km_text = km_text.str.replace(r"[^\d]", "", regex=True)

# Convert the cleaned values to numeric integers
df_clean["Kilometers_Driven"] = pd.to_numeric(
    km_text.replace("", pd.NA),
    errors="coerce"
).astype("Int64")

# Inspect the results
display(
    df_clean[["Car_Name", "Kilometers_Driven"]].head(10)
)


,Car_Name,Kilometers_Driven
0,2020 Toyota Glanza V CVT,86387
1,2021 Toyota Glanza V,105485
2,2021 Toyota Glanza G CVT,70609
3,2025 Toyota Glanza V AMT,19742
4,2018 Toyota YARIS G CVT,74307
5,2018 Toyota YARIS G CVT,92328
6,2017 Toyota Corolla Altis GL DIESEL,141572
7,2022 Toyota URBAN CRUISER HYRYDER V HYBRID,108490
8,2025 Toyota Urban Cruiser Taisor S 1.2 Petrol AMT,8207
9,2013 Toyota Etios GD,95534


In [ ]:

def clean_price_to_rupees(value):
    """Convert a displayed Indian car price into numeric rupees."""

    # Preserve missing values
    if pd.isna(value):
        return np.nan

    # Numeric values from our current scraper are already in rupees
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    # Remove the rupee symbol, commas and surrounding spaces
    price_text = re.sub(r"[₹,\s]", "", str(value)).lower()

    # Extract the number and optional unit
    match = re.fullmatch(
        r"(\d+(?:\.\d+)?)(lakh|lac|crore|cr)?",
        price_text
    )

    if not match:
        return np.nan

    amount = float(match.group(1))
    unit = match.group(2)

    # Convert lakh and crore to rupees
    if unit in ["lakh", "lac"]:
        amount *= 100_000
    elif unit in ["crore", "cr"]:
        amount *= 10_000_000

    return amount


# Apply the cleaning function
df_clean["Price"] = df_clean["Price"].apply(clean_price_to_rupees)

# Inspect the numeric prices
display(df_clean[["Car_Name", "Price"]].head(10))
print("Price data type:", df_clean["Price"].dtype)


,Car_Name,Price
0,2020 Toyota Glanza V CVT,534000.0
1,2021 Toyota Glanza V,551000.0
2,2021 Toyota Glanza G CVT,555000.0
3,2025 Toyota Glanza V AMT,924000.0
4,2018 Toyota YARIS G CVT,531000.0
5,2018 Toyota YARIS G CVT,482000.0
6,2017 Toyota Corolla Altis GL DIESEL,655000.0
7,2022 Toyota URBAN CRUISER HYRYDER V HYBRID,1247000.0
8,2025 Toyota Urban Cruiser Taisor S 1.2 Petrol AMT,1471000.0
9,2013 Toyota Etios GD,332000.0


Price data type: float64


In [ ]:

# Normalize fuel labels using a Regex pattern
fuel_pattern = r"\b(Petrol|Diesel|CNG|Electric|Hybrid)\b"

df_clean["Fuel_Type"] = (
    df_clean["Fuel_Type"]
    .astype("string")
    .str.extract(fuel_pattern, flags=re.IGNORECASE, expand=False)
    .str.title()
)

# Normalize transmission labels
transmission_text = df_clean["Transmission"].astype("string")

df_clean["Transmission"] = np.select(
    [
        transmission_text.str.contains(
            r"\b(auto|automatic|amt|cvt|dct|dsg)\b",
            case=False,
            regex=True,
            na=False
        ),
        transmission_text.str.contains(
            r"\b(manual|mt)\b",
            case=False,
            regex=True,
            na=False
        )
    ],
    ["Automatic", "Manual"],
    default="Unknown"
)

# Inspect the category counts
print("Fuel types:")
display(df_clean["Fuel_Type"].value_counts(dropna=False))

print("\nTransmission types:")
display(df_clean["Transmission"].value_counts(dropna=False))


Fuel types:


/tmp/ipykernel_473/1193850722.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  transmission_text.str.contains(
/tmp/ipykernel_473/1193850722.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  transmission_text.str.contains(


,count
Fuel_Type,
Petrol,194
Diesel,31
Hybrid,4
Cng,3



Transmission types:


,count
Transmission,
Manual,146
Automatic,86


In [ ]:

# Remove exact duplicate rows
before = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

print("Exact duplicates removed:", before - len(df_clean))

# Identify missing values in important numeric columns
required_numeric = [
    "Model_Year",
    "Kilometers_Driven",
    "Price"
]

print("\nMissing required numeric values:")
display(df_clean[required_numeric].isnull().sum())

# Flag records that need review instead of silently deleting them
df_clean["Needs_Review"] = df_clean[required_numeric].isnull().any(axis=1)

print("\nRecords requiring review:", df_clean["Needs_Review"].sum())

# Owner type is not currently available from our extraction method.
# Keep it missing until a reliable source field is identified.
print(
    "\nOwner type values:",
    df_clean["Owner_Type"].value_counts(dropna=False).to_dict()
)


Exact duplicates removed: 0

Missing required numeric values:


,0
Model_Year,0
Kilometers_Driven,0
Price,0



Records requiring review: 0

Owner type values: {None: 232}


In [ ]:

# Use the current project year for age calculations
analysis_year = 2026

# Feature 1: Car age
df_clean["Car_Age"] = analysis_year - df_clean["Model_Year"]

# Remove impossible negative ages
df_clean.loc[df_clean["Car_Age"] < 0, "Car_Age"] = np.nan

# Feature 2: Mileage category
df_clean["KM_Category"] = pd.cut(
    df_clean["Kilometers_Driven"],
    bins=[-1, 50_000, 100_000, 150_000, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)

# Feature 3: Age category
df_clean["Age_Category"] = pd.cut(
    df_clean["Car_Age"],
    bins=[-1, 3, 6, 10, np.inf],
    labels=["New", "Moderate", "Old", "Very Old"]
)

# Feature 4: Price category in rupees
df_clean["Price_Category"] = pd.cut(
    df_clean["Price"],
    bins=[0, 500_000, 1_000_000, 2_000_000, np.inf],
    labels=["Budget", "Mid Range", "Premium", "Luxury"],
    include_lowest=True
)

# Feature 5: Price per 10,000 kilometres
df_clean["Price_per_10k_KM"] = np.where(
    df_clean["Kilometers_Driven"] > 0,
    df_clean["Price"] / df_clean["Kilometers_Driven"] * 10_000,
    np.nan
)

# Display the new features
display(
    df_clean[
        [
            "Car_Name",
            "Car_Age",
            "KM_Category",
            "Age_Category",
            "Price_Category",
            "Price_per_10k_KM"
        ]
    ].head(10)
)


,Car_Name,Car_Age,KM_Category,Age_Category,Price_Category,Price_per_10k_KM
0,2020 Toyota Glanza V CVT,6.0,Medium,Moderate,Mid Range,6.181486e+04
1,2021 Toyota Glanza V,5.0,High,Moderate,Mid Range,5.223491e+04
2,2021 Toyota Glanza G CVT,5.0,Medium,Moderate,Mid Range,7.860188e+04
3,2025 Toyota Glanza V AMT,1.0,Low,New,Mid Range,4.680377e+05
4,2018 Toyota YARIS G CVT,8.0,Medium,Old,Mid Range,7.146029e+04
5,2018 Toyota YARIS G CVT,8.0,Medium,Old,Budget,5.220518e+04
6,2017 Toyota Corolla Altis GL DIESEL,9.0,High,Old,Mid Range,4.626621e+04
7,2022 Toyota URBAN CRUISER HYRYDER V HYBRID,4.0,High,Moderate,Premium,1.149415e+05
8,2025 Toyota Urban Cruiser Taisor S 1.2 Petrol AMT,1.0,Low,New,Premium,1.792372e+06
9,2013 Toyota Etios GD,13.0,Medium,Very Old,Budget,3.475203e+04


In [ ]:

# Verify that the cleaned dataset has the expected columns
print("Cleaned dataset shape:", df_clean.shape)

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values:")
display(df_clean.isnull().sum().to_frame("Missing_Count"))

print("\nPreview:")
display(df_clean.head(10))

# Save the cleaned dataset
df_clean.to_csv(
    "cleaned_used_cars.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: cleaned_used_cars.csv")


Cleaned dataset shape: (232, 17)

Data types:
Car_Name             string[python]
Brand                        object
City                         object
Model_Year                    int64
Kilometers_Driven             Int64
Fuel_Type            string[python]
Transmission                 object
Price                       float64
Owner_Type                   object
Location                     object
URL                          object
Needs_Review                   bool
Car_Age                     float64
KM_Category                category
Age_Category               category
Price_Category             category
Price_per_10k_KM            float64
dtype: object

Missing values:


,Missing_Count
Car_Name,0
Brand,0
City,0
Model_Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Price,0
Owner_Type,232
Location,0



Preview:


,Car_Name,Brand,City,Model_Year,Kilometers_Driven,Fuel_Type,Transmission,Price,Owner_Type,Location,URL,Needs_Review,Car_Age,KM_Category,Age_Category,Price_Category,Price_per_10k_KM
0,2020 Toyota Glanza V CVT,Toyota,Hyderabad,2020,86387,Petrol,Automatic,534000.0,None,"GVK One Mall, Banjara Hills, Hyderabad, Telangana",https://www.cars24.com/buy-used-toyota-cars-hy...,False,6.0,Medium,Moderate,Mid Range,6.181486e+04
1,2021 Toyota Glanza V,Toyota,Hyderabad,2021,105485,Petrol,Manual,551000.0,None,"Upperpally, Attapur, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,5.0,High,Moderate,Mid Range,5.223491e+04
2,2021 Toyota Glanza G CVT,Toyota,Hyderabad,2021,70609,Petrol,Automatic,555000.0,None,"Bachupally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,5.0,Medium,Moderate,Mid Range,7.860188e+04
3,2025 Toyota Glanza V AMT,Toyota,Hyderabad,2025,19742,Petrol,Automatic,924000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,1.0,Low,New,Mid Range,4.680377e+05
4,2018 Toyota YARIS G CVT,Toyota,Hyderabad,2018,74307,Petrol,Automatic,531000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,8.0,Medium,Old,Mid Range,7.146029e+04
5,2018 Toyota YARIS G CVT,Toyota,Hyderabad,2018,92328,Petrol,Automatic,482000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,8.0,Medium,Old,Budget,5.220518e+04
6,2017 Toyota Corolla Altis GL DIESEL,Toyota,Hyderabad,2017,141572,Diesel,Manual,655000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,9.0,High,Old,Mid Range,4.626621e+04
7,2022 Toyota URBAN CRUISER HYRYDER V HYBRID,Toyota,Hyderabad,2022,108490,Hybrid,Automatic,1247000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,4.0,High,Moderate,Premium,1.149415e+05
8,2025 Toyota Urban Cruiser Taisor S 1.2 Petrol AMT,Toyota,Hyderabad,2025,8207,Petrol,Automatic,1471000.0,None,"Bachupally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,1.0,Low,New,Premium,1.792372e+06
9,2013 Toyota Etios GD,Toyota,Hyderabad,2013,95534,Diesel,Manual,332000.0,None,"Kompally, Hyderabad",https://www.cars24.com/buy-used-toyota-cars-hy...,False,13.0,Medium,Very Old,Budget,3.475203e+04



Saved: cleaned_used_cars.csv


In [ ]:

from google.colab import files

files.download("cleaned_used_cars.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>